# Archetype Profiles

Interpret the archetypes by looking at:

- The mean feature vector per archetype (heatmap)
- A radar chart of `pct_total_<event_type>` per archetype
- The within-event-type cluster mix per archetype
- A few example players per archetype

Use this notebook to give each archetype a human-readable name
(e.g. *CD-2 = ball-playing center-back*).

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

# locate repo root from the notebook's location
HERE = Path.cwd()
ROOT = HERE
while ROOT != ROOT.parent and not (ROOT / "configs" / "config.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

CFG = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
ART = ROOT / CFG["data_paths"]["archetype_artifacts"]
EVENT_MODEL_DIR = ART / "event_clusters"
ARCH_MODEL_DIR = ART / "archetype_clusters"

EVENT_TYPES_PER_GROUP = CFG["event_clustering"]["event_types_per_group"]
N_CLUSTERS_PER_EVENT = CFG["event_clustering"]["n_clusters_per_event"]
N_ARCHETYPES_PER_GROUP = CFG["archetype_clustering"]["n_archetypes_per_group"]
POSITION_GROUPS = list(EVENT_TYPES_PER_GROUP.keys())

print("ROOT:", ROOT)
print("ART :", ART)

In [ ]:
from archetypes.player_aggregation import feature_columns_for_group

feats = pd.read_parquet(ART / "player_features.parquet")
amap  = pd.read_parquet(ART / "player_archetype_map.parquet")
df = feats.merge(amap[["player_id", "archetype"]], on="player_id")
player_info = pd.read_parquet(ROOT / "data" / "raw" / "player_info.parquet")
player_info = player_info.rename(columns={"wyId": "player_id"})
df = df.merge(player_info[["player_id", "shortName"]], on="player_id", how="left")
print(df.shape)
df.head()

## Heatmap of mean feature values per archetype

In [ ]:
for g in POSITION_GROUPS:
    cols = feature_columns_for_group(EVENT_TYPES_PER_GROUP[g], N_CLUSTERS_PER_EVENT[g])
    sub = df[df["position_group"] == g]
    prof = sub.groupby("archetype")[cols].mean()
    fig, ax = plt.subplots(figsize=(0.9 * len(prof) + 4, 0.32 * len(cols) + 1))
    im = ax.imshow(prof.T.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(prof.index))); ax.set_xticklabels(prof.index)
    ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8)
    fig.colorbar(im, ax=ax)
    ax.set_title(f"{g} — mean features per archetype")
    plt.tight_layout(); plt.show()

## Radar charts: `pct_total_<event_type>` per archetype

In [ ]:
def radar(ax, labels, values, title):
    n = len(labels)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
    values = list(values) + [values[0]]
    angles = angles + [angles[0]]
    ax.plot(angles, values, lw=1.5)
    ax.fill(angles, values, alpha=0.2)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=9, pad=10)

for g in POSITION_GROUPS:
    ets = EVENT_TYPES_PER_GROUP[g]
    sub = df[df["position_group"] == g]
    archetypes = sorted(sub["archetype"].unique())
    fig, axes = plt.subplots(1, len(archetypes),
                             figsize=(3 * len(archetypes), 3),
                             subplot_kw=dict(polar=True))
    if len(archetypes) == 1:
        axes = [axes]
    # use a shared scale so radars are comparable within the group
    cols = [f"pct_total_{et}" for et in ets]
    ymax = sub[cols].mean().max() * 1.5
    for ax, arch in zip(axes, archetypes):
        vals = sub.loc[sub["archetype"] == arch, cols].mean().values
        radar(ax, ets, vals, f"{arch}  (n={(sub['archetype']==arch).sum()})")
        ax.set_ylim(0, ymax)
    fig.suptitle(f"{g}: event-type mix per archetype", y=1.05)
    fig.tight_layout(); plt.show()

## Within-event-type cluster mix per archetype

In [ ]:
for g in POSITION_GROUPS:
    ets = EVENT_TYPES_PER_GROUP[g]
    sub = df[df["position_group"] == g]
    archetypes = sorted(sub["archetype"].unique())
    fig, axes = plt.subplots(1, len(ets), figsize=(3.2 * len(ets), 3.2), sharey=True)
    if len(ets) == 1:
        axes = [axes]
    for ax, et in zip(axes, ets):
        cols = [f"pct_cluster_{et}_{i}" for i in range(N_CLUSTERS_PER_EVENT[g][et])]
        M = sub.groupby("archetype")[cols].mean().reindex(archetypes)
        bottom = np.zeros(len(M))
        for i, c in enumerate(cols):
            ax.bar(M.index, M[c].values, bottom=bottom,
                   label=f"c{i}")
            bottom = bottom + M[c].values
        ax.set_title(et, fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=20)
    axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    fig.suptitle(f"{g}: within-event-type cluster mix per archetype", y=1.04)
    fig.tight_layout(); plt.show()

## Example players per archetype

In [ ]:
for g in POSITION_GROUPS:
    sub = df[df["position_group"] == g]
    print(f"=== {g} ===")
    for arch in sorted(sub["archetype"].unique()):
        names = sub.loc[sub["archetype"] == arch, "shortName"].dropna().sample(
            min(8, (sub["archetype"] == arch).sum()), random_state=0).tolist()
        print(f"  {arch}: " + ", ".join(names))
    print()